# 32. UI 학습 노트 — 사이드바 질문 기록 (2026-11-09)

사이드바에 "이번 접속에서 한 질문" 목록을 보여 주고, 누르면 그 질문과 **저장돼 있는 답변**만 보는 기능이다.
실제 코드는 `src/rag/ui/app.py`, 시험은 `tests/rag/ui/test_app_buttons.py` 에 있다.

## 원복 방법
| 방법 | 명령 |
|---|---|
| 로컬 태그 | `git checkout pre-history-20261109 -- src/rag/ui/app.py` (이 기능 직전 커밋 d986e5c, 푸시된 시안 C 상태) |
| 백업 파일 | `notebooks/prevention_legal/backups/app_before_history_20261109.py` (Git 제외, 이 컴퓨터에만 있음) |
| 시험 | `test_app_buttons.py` 끝의 `사이드바 질문 기록` 구간 시험 5개를 함께 뺀다 |

## 설계 결정 (사용자 선택)
- **새 저장소를 쓰지 않는다.** 화면 대화(`session_state["history"]`)를 그대로 쓴다. 그래서 새로고침하면 사라진다.
- **서버의 질문 이력 파일(`queries.jsonl`)과 별개다.** 그 파일은 모든 접속자의 질문이 쌓이는 점검용 로그라 사용자에게 보이면 안 된다.
- **누르면 질문을 다시 실행하지 않는다.** 저장돼 있는 답변을 다시 보여 줄 뿐이라 LLM 호출이 없고 답이 달라지지 않는다.

## 이 노트에서 배우는 것
1. 목록의 각 항목을 구분할 **번호(hid)** 를 질문을 넣을 때 한 번 만든다.
2. 고른 항목은 `session_state[FOCUS_KEY]` 에 맡기고, 그리는 쪽(`render_history`)이 그 번호만 걸러 보여 준다.
3. 새 질문이 오면 고른 표시를 풀어 새 답이 바로 보이게 한다.

In [ ]:
import uuid

FOCUS_KEY = "focus_hid"   # 사이드바에서 고른 질문의 번호. 없으면 전체 대화를 보여 준다


# ── 1) 질문을 넣을 때 번호(hid)를 붙인다 (main 안) ───────────────────────
def make_item(question, result):
    """history 에 넣을 한 건. result 는 ask() 의 결과(또는 {"error": ...}).
    hid: 사이드바 버튼과 본문을 짝짓는 번호. request_id 는 오류 항목에는 없어서 따로 만든다."""
    return {"question": question, "hid": uuid.uuid4().hex[:8], **result}


# ── 2) 버튼 글자 ────────────────────────────────────────────────────
STATUS_ICONS = {"ANSWERED": "✅", "PARTIAL": "🟡", "INSUFFICIENT_EVIDENCE": "⚪", "ERROR": "⛔"}


def history_label(item):
    """상태 아이콘 + 질문 앞부분(줄바꿈·공백은 한 칸으로). 전체 질문은 버튼 도움말(help)에 둔다."""
    icon = "⛔" if "error" in item else STATUS_ICONS.get(item["output"]["status"], "•")
    text = " ".join(item["question"].split())          # split() 후 join: 연속 공백·줄바꿈을 한 칸으로
    return f"{icon} {text[:26]}{'…' if len(text) > 26 else ''}"


# ── 3) 그리는 쪽: 고른 번호가 있으면 그 질문 하나만 ──────────────────────
def pick_shown(items, focus):
    """render_history 의 걸러내기 로직만 따로 뽑은 예시.
    focus 가 있고 목록에 있으면 그 하나, 없는 번호이거나 focus 가 없으면 전체(최근 10건)."""
    shown = items[:10]
    chosen = [i for n, i in enumerate(shown) if (i.get("hid") or f"i{n}") == focus] if focus else []
    return chosen or shown          # chosen 이 비면(없는 번호) 전체로 되돌아가 빈 화면이 되지 않는다


demo = [make_item(f"질문 {n}", {"output": {"status": "ANSWERED"}}) for n in range(3)]
print([history_label(i) for i in demo])
print(len(pick_shown(demo, demo[1]["hid"])), len(pick_shown(demo, None)), len(pick_shown(demo, "없는번호")))

## 4) Streamlit 코드 흐름 (app.py)
```python
def pick_history(hid):                 # 사이드바 버튼의 on_click 콜백: 번호만 맡겨 둔다(질문 재실행 없음)
    st.session_state[FOCUS_KEY] = hid

def show_history_sidebar(items):
    with st.sidebar:
        for number, item in enumerate(items[:10]):
            hid = item.get("hid") or f"i{number}"
            st.button(history_label(item), key=f"hist-pick-{hid}", on_click=pick_history, args=(hid,),
                      help=item["question"], type="primary" if hid == focus else "secondary", width="stretch")
```
- 고른 항목은 `type="primary"` 로 진하게 보인다.
- 고른 상태에서는 사이드바와 본문에 `전체 대화 보기` 버튼이 나온다(사이드바가 접힌 좁은 화면 때문에 본문에도 둔다).
- `현재 화면 대화 지우기` 는 `history` 와 고른 표시를 함께 지운다. 저장한 답변은 그대로다.

## 5) 시험 (`test_app_buttons.py`)
| 시험 | 확인하는 약속 |
|---|---|
| 새것부터 목록 | 처음엔 "아직 질문이 없습니다", 질문 둘 뒤에는 둘째·첫 순서로 상태 아이콘과 함께 나온다 |
| 선택 | 오래된 질문을 누르면 그 답만 보이고 `ask` 호출 수가 늘지 않는다. 전체 보기로 돌아가면 둘 다 보인다 |
| 새 질문·지우기 | 새 질문이면 고른 표시가 풀린다. 대화 지우기 뒤에는 사이드바가 비어 있다 |
| 라벨 | 긴 질문은 `…` 로 줄이고 전체 질문은 help 에 둔다 |
| 개발자 화면 | `개발자 설정` 헤더와 질문 기록이 같은 사이드바에 있다 |

## 6) 직접 해 보기
1. 목록을 10건에서 5건으로 줄이려면 어디를 바꿔야 할까? (`items[:10]` 이 두 곳에 있다. 하나만 바꾸면 어떤 불일치가 생길까?)
2. 같은 질문을 두 번 했을 때 목록에는 어떻게 보이는가? 같은 질문을 하나로 묶으려면 어떤 키를 써야 할까? (`history.normalize_question` 이 있다.)
3. 고른 질문이 목록에서 밀려나 11번째가 되면 화면은 어떻게 되는가? `pick_shown` 의 마지막 줄(`chosen or shown`)이 왜 필요한가?
4. (생각해 보기) 새로고침 후에도 기록을 남기고 싶다면 어떤 저장소가 필요하고, 질문 원문의 개인정보 문제는 어떻게 다뤄야 할까?

## 7) 한계 (사실만 기록)
- 새로고침·접속 종료 시 기록이 사라진다(의도된 동작).
- 목록은 최근 10건뿐이고, 11번째 이전 질문은 사이드바에서 열 수 없다.
- Chrome(폭 1280·400)에서 목록·선택·전체 보기를 확인했다. 개발자 화면의 사이드바는 AppTest로만 확인했다.